# 🚀 Pinterest Realism Engine — Free Cloud GPU AI Upscaler
**Run 4K Real-ESRGAN Super-Resolution on a 100% Free Google Colab GPU (No Laptop RAM or GPU used!)**

### Instructions:
1. Make sure GPU is enabled: Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Save.
2. Click the **Play (Run)** button on **Cell 1** below.
3. It will print your public `https://xxxx.trycloudflare.com` URL.
4. Copy that URL and paste it into your local `.env` file as:
   `COLAB_UPSCALER_URL=https://xxxx.trycloudflare.com`

In [ ]:
# ==============================================================================
# CELL 1: SETUP & START AI UPSCALER CLOUD SERVER
# ==============================================================================
import subprocess, sys, os, io, time, re, threading

print("📦 [1/4] Installing FastAPI, Uvicorn & Spandrel (Modern AI Engine)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "spandrel", "pillow"], check=True)

print("🌐 [2/4] Setting up Cloudflare tunnel...")
subprocess.run(["curl", "-s", "-L", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", "/usr/local/bin/cloudflared"], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

print("🧠 [3/4] Downloading Real-ESRGAN x4plus AI weights (64MB)...")
if not os.path.exists("RealESRGAN_x4plus.pth"):
    subprocess.run(["curl", "-s", "-L", "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth", "-o", "RealESRGAN_x4plus.pth"], check=True)

import torch
import torchvision.transforms.functional as TF
from PIL import Image
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import Response
import uvicorn
from spandrel import ModelLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Loading model on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Warning: GPU not active!)'}")

model_loader = ModelLoader()
upscale_model = model_loader.load_from_file("RealESRGAN_x4plus.pth").to(device).eval()
if device.type == "cuda":
    upscale_model = upscale_model.half()

print("✅ Real-ESRGAN Model loaded into GPU VRAM successfully!")

app = FastAPI(title="Pinterest Realism Engine AI Upscaler")

@app.get("/")
def health_check():
    return {
        "status": "online",
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "half_precision": device.type == "cuda"
    }

@app.post("/upscale")
async def upscale_endpoint(file: UploadFile = File(...)):
    raw_bytes = await file.read()
    input_image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    tensor = TF.to_tensor(input_image).unsqueeze(0).to(device)
    if device.type == "cuda":
        tensor = tensor.half()
    with torch.no_grad():
        output_tensor = upscale_model(tensor).clamp(0, 1)
    output_image = TF.to_pil_image(output_tensor.squeeze(0).float().cpu())
    output_buf = io.BytesIO()
    output_image.save(output_buf, format="JPEG", quality=98, subsampling=0, optimize=True)
    return Response(content=output_buf.getvalue(), media_type="image/jpeg")

def start_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=start_api, daemon=True).start()
time.sleep(2)

print("🚀 [4/4] Starting Cloudflare Public Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 AI UPSCALER IS 100% ONLINE AND READY ON FREE COLAB GPU!")
    print(f"👉 Public Cloudflare URL: {tunnel_url}")
    print("\n👉 In your local .env, set:")
    print(f"   COLAB_UPSCALER_URL={tunnel_url}")
    print("="*70 + "\n")
else:
    print("⚠️ Could not automatically extract tunnel URL.")

# Keep alive
while True:
    time.sleep(60)


In [ ]:
# ==============================================================================
# CELL 2 (OPTIONAL): SELF-TEST & PREVIEW UPSCALE INSIDE COLAB
# ==============================================================================
# Run this cell to test upscaling directly inside Colab and see the results!
import requests, io
from PIL import Image
import matplotlib.pyplot as plt

# Download a sample test image
sample_url = "https://raw.githubusercontent.com/xinntao/Real-ESRGAN/master/inputs/0014.jpg"
resp = requests.get(sample_url)
test_img = Image.open(io.BytesIO(resp.content))

# Send to local API
print(f"Original Image Size: {test_img.size}")
files = {"file": ("test.jpg", resp.content, "image/jpeg")}
up_resp = requests.post("http://127.0.0.1:8000/upscale", files=files)
upscaled_img = Image.open(io.BytesIO(up_resp.content))
print(f"🎉 Upscaled 4K Size: {upscaled_img.size}")

# Display side-by-side
fig, axs = plt.subplots(1, 2, figsize=(14, 7))
axs[0].imshow(test_img)
axs[0].set_title(f"Original: {test_img.size}")
axs[0].axis("off")
axs[1].imshow(upscaled_img)
axs[1].set_title(f"AI 4K Master: {upscaled_img.size}")
axs[1].axis("off")
plt.show()
